# Merge TCAV Outputs

This notebook loads all per-system TCAV CSV files from one output run, concatenates them into one long dataframe, and saves the merged CSV for downstream analysis.


In [1]:
from pathlib import Path

import pandas as pd


In [2]:
# ===== Config =====
PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice')
TCAV_ROOT = PROJECT_ROOT / 'redimnet' / 'tcav' / 'captum_tcav' / 'asvspoof5'
OUTPUT_SUBDIR = 'subset_20spk_20utts_per_spk_one_logistic_head'
OUTPUT_DIR = TCAV_ROOT / 'output' / OUTPUT_SUBDIR

# Set to True if you want to drop A12 immediately.
EXCLUDE_A12 = False

MERGED_CSV_NAME = 'merged_tcav_long.csv'

print('OUTPUT_DIR =', OUTPUT_DIR)
print('Exists =', OUTPUT_DIR.exists())


OUTPUT_DIR = /home/SpeakerRec/BioVoice/redimnet/tcav/captum_tcav/asvspoof5/output/subset_20spk_20utts_per_spk_one_logistic_head
Exists = True


In [3]:
csv_paths = sorted(
    p for p in OUTPUT_DIR.glob('tcav_*.csv')
    if p.is_file() and not p.name.endswith('_wide.csv')
)

print('Found CSV files:', len(csv_paths))
for p in csv_paths[:10]:
    print(' ', p.name)
if len(csv_paths) > 10:
    print(' ...')


Found CSV files: 32
  tcav_dev_A09_bonafide.csv
  tcav_dev_A09_spoof.csv
  tcav_dev_A10_bonafide.csv
  tcav_dev_A10_spoof.csv
  tcav_dev_A11_bonafide.csv
  tcav_dev_A11_spoof.csv
  tcav_dev_A12_bonafide.csv
  tcav_dev_A12_spoof.csv
  tcav_dev_A13_bonafide.csv
  tcav_dev_A13_spoof.csv
 ...


In [4]:
required_cols = {
    'utt_id',
    'speaker_id',
    'split',
    'source_partition',
    'system_id',
    'target_class',
    'concept_name',
    'sign_count',
    'magnitude',
}

frames = []
for path in csv_paths:
    df = pd.read_csv(path)
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'{path.name} is missing columns: {sorted(missing)}')
    df['source_file'] = path.name
    frames.append(df)

merged_df = pd.concat(frames, axis=0, ignore_index=True)

if EXCLUDE_A12:
    merged_df = merged_df[merged_df['system_id'] != 'A12'].copy()

merged_df = merged_df.sort_values(
    ['source_partition', 'system_id', 'target_class', 'speaker_id', 'utt_id', 'concept_name']
).reset_index(drop=True)

print('Merged rows =', len(merged_df))
print('Merged utterances =', merged_df['utt_id'].nunique())
print('Systems =', sorted(merged_df['system_id'].unique().tolist()))
print('Target classes =', sorted(merged_df['target_class'].unique().tolist()))


Merged rows = 153600
Merged utterances = 8307
Systems = ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16']
Target classes = ['bonafide', 'spoof']


In [5]:
summary_df = pd.DataFrame({
    'rows': [len(merged_df)],
    'utterances': [merged_df['utt_id'].nunique()],
    'speakers': [merged_df['speaker_id'].nunique()],
    'systems': [merged_df['system_id'].nunique()],
    'concepts': [merged_df['concept_name'].nunique()],
})
display(summary_df)

display(
    merged_df.groupby(['system_id', 'target_class'])['utt_id']
    .nunique()
    .rename('n_utterances')
    .reset_index()
    .sort_values(['system_id', 'target_class'])
)


,rows,utterances,speakers,systems,concepts
0,153600,8307,81,16,12


,system_id,target_class,n_utterances
0,A01,bonafide,400
1,A01,spoof,400
2,A02,bonafide,400
3,A02,spoof,400
4,A03,bonafide,400
5,A03,spoof,400
6,A04,bonafide,400
7,A04,spoof,400
8,A05,bonafide,400
9,A05,spoof,400


In [6]:
out_path = OUTPUT_DIR / MERGED_CSV_NAME
merged_df.to_csv(out_path, index=False)
print('Saved merged CSV to:', out_path)


Saved merged CSV to: /home/SpeakerRec/BioVoice/redimnet/tcav/captum_tcav/asvspoof5/output/subset_20spk_20utts_per_spk_one_logistic_head/merged_tcav_long.csv
